# Exploration real time  - Trip_updates
### D’apres la discussion de groupe, nous avons exploré les premieres donnees real time:  https://ara-api.enroute.mobi/rla/gtfs/trip-updates

In [6]:
from google.transit import gtfs_realtime_pb2
import requests
import pandas as pd
from datetime import datetime

In [5]:
from google.transit import gtfs_realtime_pb2
import requests

feed = gtfs_realtime_pb2.FeedMessage()
response = requests.get('https://ara-api.enroute.mobi/rla/gtfs/trip-updates')
feed.ParseFromString(response.content)
for entity in feed.entity[:2]:
  if entity.HasField('trip_update'):
    print(entity.trip_update)
    

trip {
  trip_id: "6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semaine-04"
  route_id: "L1"
  direction_id: 1
}
stop_time_update {
  stop_sequence: 7
  arrival {
    time: 1756898923
  }
  departure {
    time: 1756898923
  }
  stop_id: "2510"
}
stop_time_update {
  stop_sequence: 8
  arrival {
    time: 1756899074
  }
  departure {
    time: 1756899074
  }
  stop_id: "2514"
}
stop_time_update {
  stop_sequence: 9
  arrival {
    time: 1756899194
  }
  departure {
    time: 1756899194
  }
  stop_id: "2522"
}
stop_time_update {
  stop_sequence: 10
  arrival {
    time: 1756899330
  }
  departure {
    time: 1756899330
  }
  stop_id: "2525"
}
stop_time_update {
  stop_sequence: 11
  arrival {
    time: 1756899450
  }
  departure {
    time: 1756899450
  }
  stop_id: "2505"
}
stop_time_update {
  stop_sequence: 12
  arrival {
    time: 1756899543
  }
  departure {
    time: 1756899543
  }
  stop_id: "1407"
}

trip {
  trip_id: "6422640-99_A_50_9901_13:58-SETP2025-99-Semaine-11"
  route_id: "9

### Premier analyse visuel, il y a certains champs qui manquent parfois, arrival et departure time. aussi route_id pourrait être util pour faire merge avec trips ou routes (statiques)

In [22]:

for entity in feed.entity [:5]:
    if entity.HasField('trip_update'):
        trip = entity.trip_update.trip

        print(trip.trip_id)
        print(trip.route_id)
        print(trip.direction_id)
        
    

6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semaine-04
L1
1
6422640-99_A_50_9901_13:58-SETP2025-99-Semaine-11
99
1
5097935-69_A_40_6901_15:25-PROJET2023-69-Semaine-15
69
1
6446945-05_A_98_0507_14:51-SETP2025-05-Mercredi-38
05
1
6257240-63_A_48_6302_15:10-PROJET2025-63-Mercredi-73
63
1


In [ ]:
# verification de presence de arrival et departure

for stop_time_update in entity.trip_update.stop_time_update[:5]:
    if stop_time_update.HasField("arrival") and stop_time_update.arrival.HasField("time"):
        print(stop_time_update.arrival.time)

    if stop_time_update.HasField("departure") and stop_time_update.departure.HasField("time"):
        print(stop_time_update.departure.time)



1756905000
1756905240
1756905240
1756905253
1756905253
1756905344
1756905344
1756905365
1756905365


In [32]:
# Construction pour le dataframe global avec correction pour direction_id

trips_update_list = []

for entity in feed.entity:
    if entity.HasField('trip_update'):
        trip = entity.trip_update.trip

        trip_id = trip.trip_id
        route_id = trip.route_id
        direction_id = trip.direction_id if trip.HasField("direction_id") else None

        for stop_time_update in entity.trip_update.stop_time_update:
            stop_id = stop_time_update.stop_id
            stop_sequence = stop_time_update.stop_sequence

            if stop_time_update.HasField("arrival") and stop_time_update.arrival.HasField("time"):
                arrival_time = stop_time_update.arrival.time
            else:
                arrival_time = None

            if stop_time_update.HasField("departure") and stop_time_update.departure.HasField("time"):
                departure_time = stop_time_update.departure.time
            else:
                departure_time = None

            trips_update_list.append({
                "trip_id": trip_id,
                "route_id": route_id,
                "direction_id": direction_id,
                "stop_id": stop_id,
                "stop_sequence": stop_sequence,
                "arrival_time": arrival_time,
                "departure_time": departure_time
            })




In [33]:
df = pd.DataFrame(trips_update_list)
df.head()

,trip_id,route_id,direction_id,stop_id,stop_sequence,arrival_time,departure_time
0,6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semain...,L1,1.0,2510,7,1.756899e+09,1.756899e+09
1,6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semain...,L1,1.0,2514,8,1.756899e+09,1.756899e+09
2,6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semain...,L1,1.0,2522,9,1.756899e+09,1.756899e+09
3,6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semain...,L1,1.0,2525,10,1.756899e+09,1.756899e+09
4,6452805-T1_A_12_VT111_13:18-SETP2025-L1-Semain...,L1,1.0,2505,11,1.756899e+09,1.756899e+09


### D'apres comprehension de brief, discussion de group et avancement cet procesus d'exploration servira pour creer le DAG pour trips_update avec plusier taches selon l'structure general a definir